In [397]:
from typing import override
from torch import Tensor, nn
import torch
import numpy as np


# Network hyperparam
dim = 65
L = 4
mode = 8
width = 32
lr = 5e-4
du = 3  # 2 for velociy x, y and pressure.

NU_LAYER = 4
BCU0_LAYER = 5 
BCU1_LAYER = 6
da = du + 4

# NN
# Fourier Convolution Operator
class FCO(nn.Module):
    def __init__(self, mode: int, width: int) -> None:
        super().__init__()
        self.mode: int = mode
        
        self.R = nn.Parameter(torch.randn(mode, width, width, dtype=torch.complex64))

    @override
    def forward(self, x: Tensor) -> Tensor:
        # x of shape: (B, dimT, dim, dim, width)
        shape = x.shape
        x_hat = torch.fft.fftn(x, dim=(1, 2))[:, :self.mode, :self.mode, :]
        x_hat = torch.einsum('mij,btmni->btmnj', self.R, x_hat)
        x_hat = torch.fft.ifftn(x_hat, s=shape)
        x = x_hat.real
        return x


class PINOLayer(nn.Module):
    def __init__(self, mode: int, width: int) -> None:
        super().__init__()

        self.k: nn.Module = FCO(mode, width)
        self.w: nn.Module = nn.Linear(width, width)
        self.sigma: nn.Module = nn.GELU()

    @override
    def forward(self, x: Tensor) -> Tensor:
        w = self.w(x)
        k = self.k(x)
        x = self.sigma(w + k)
        return x


class PINO(nn.Module):
    def __init__(self, da: int, du: int, mode: int, width: int, L: int) -> None:
        super().__init__()

        # uplift
        self.P = nn.Linear(da, width)
        # projection
        self.Q = nn.Linear(width, du)

        # layers = [PINOLayer(mode, width) for _ in range(L)]
        # self.core: nn.Module = nn.Sequential(
        #     P,
        #     # *layers,
        #     Q,
        # )

    @override
    def forward(self, x: Tensor) -> Tensor:
        # return self.core(x)
        x = self.P(x)
        x = self.Q(x)
        return x




# PDE param
T = (5, 10)
dimT = 50
alpha = 1
beta = 1

# re & nu not fixed

k = 1
dt = (T[1] - T[0]) / dimT
dx = 1.0 / dim

# Todo: fill in the choosen times
t = torch.arange(*T, dt)
x = torch.arange(0.0, 1.0, dx)
y = torch.arange(0.0, 1.0, dx)

# Training is done over elements of shape:
# batch, dimT, dim, dim, C
# We have to compute dimT elements before applying loss because pde loss needs it
# We could apply data loss more often but it would be a different scheme than L = Ldata + λLpde
# The batch dimension mixes data from different 

# To compute pde loss, we need pde params
# Dataset of shape a -> u
# Where a: (batch, dimT, dim, dim, C) with C = da (ex: du + broadcasted pde_params)
# u: (batch, dimT, dim, dim, C) with C = du

L2 = torch.nn.MSELoss(reduction="mean")


# a of shape (batch, dimT, dim, dim, da)
# PDE param embedded in da
# u  of shape (batch, dimT, dim, dim, du)

# residual loss over time
def residual_loss(a, u):
    # assumed density to be 1
    # v viscosity (nu)
    # du/dt = -(u.∇)u + v∇.∇u -∇p
    # R = du/dt + (u.∇)u + ∇p - v∇.∇u

    
    TRS = np.s_[:, 2:]  # Time restriction
    
    INT = np.s_[:, :, 1:-1, 1:-1, :]  # Interior
    XPO = np.s_[:, :,   2:, 1:-1, :]  # X Plus One
    XMO = np.s_[:, :,  :-2, 1:-1, :]  # X Minus One
    YPO = np.s_[:, :, 1:-1,   2:, :]  # Y Plus One
    YMO = np.s_[:, :, 1:-1,  :-2, :]  # Y Minus One 

    u_vel = u[..., :2]
    u_vel_t = u_vel[TRS]
    p_t = u[TRS][..., 2:2 + 1]
    
    nu = a[TRS][INT][..., NU_LAYER:NU_LAYER + 1]
    
    du_dt = (u_vel[INT][:, 2:] - u_vel[INT][:, 1:-1]) / dt
    
    du_dx = (u_vel_t[XPO] - u_vel_t[XMO]) / dx
    du_dy = (u_vel_t[YPO] - u_vel_t[YMO]) / dx
    du_dx2 = (u_vel_t[XPO] + u_vel_t[XMO] - 2 * u_vel_t[INT]) / (dx * dx)
    du_dy2 = (u_vel_t[YPO] + u_vel_t[YMO] - 2 * u_vel_t[INT]) / (dx * dx)
    dp_dx = (p_t[XPO] - p_t[XMO]) / dx
    dp_dy = (p_t[YPO] - p_t[YMO]) / dx
    
    c0 = u_vel_t[INT][..., 0] * du_dx[..., 0] + u_vel_t[INT][..., 1] * du_dy[..., 0]
    c1 = u_vel_t[INT][..., 0] * du_dx[..., 1] + u_vel_t[INT][..., 1] * du_dy[..., 1]
    convection = torch.stack([c0, c1], dim=-1)
    
    viscosity = nu * (du_dx2 + du_dy2)

    grad_p = torch.cat([dp_dx, dp_dy], dim=-1)
    
    R = du_dt + convection + grad_p - viscosity

    return torch.mean(R**2)

# boundary condition loss over time
def bc_loss(a, u):
    bcu = a[..., BCU0_LAYER:BCU1_LAYER + 1]
    u_vel = u[..., :2]

    return (
        L2(u_vel[:, :, 1:-1,    0, :], bcu[:, :, 1:-1,    0, :]) +
        L2(u_vel[:, :, 1:-1,   -1, :], bcu[:, :, 1:-1,   -1, :]) +
        L2(u_vel[:, :,    0, 1:-1, :], bcu[:, :,    0, 1:-1, :]) +
        L2(u_vel[:, :,   -1, 1:-1, :], bcu[:, :,   -1, 1:-1, :])
    )

# initial condition loss
def ic_loss(a, u):
    ic_vel = a[:, 0, :, :, :2]
    u0_vel = u[:, 0, :, :, :2]
    return L2(ic_vel, u0_vel)

# L pde is J pde as long as loss is averaged over batch
def L_pde(a, u):
    return residual_loss(a, u) + alpha * bc_loss(a, u) + beta * ic_loss(a, u)



In [403]:
pino = PINO(da, du, mode, width, 0)

In [404]:
Batch = 1
test_input = torch.rand(Batch, dimT, dim, dim, da)

In [405]:
test_input.shape

torch.Size([1, 50, 65, 65, 7])

In [406]:
output = pino(test_input)

In [407]:
output.shape

torch.Size([1, 50, 65, 65, 3])

In [408]:
loss = L_pde(test_input, output)

In [409]:
loss.backward()

In [410]:
loss

tensor(1063920.7500, grad_fn=<AddBackward0>)